25.12.3
新增保存实验结果
保存了完整的实验结果

In [1]:
import torch
import torch.nn as nn
import os
from transformers import AutoModel, AutoTokenizer
from transformers import BertModel, BertTokenizer
import random
from scipy.interpolate import interp1d
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import roc_auc_score,roc_curve
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import KFold
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
import random
import matplotlib.pyplot as plt
from torch.utils.data.sampler import SubsetRandomSampler
import numpy as np
import torch
import csv
import pandas as pd
import pickle
from sklearn.metrics import roc_auc_score, roc_curve, accuracy_score, matthews_corrcoef, precision_score, recall_score, f1_score
from sklearn.metrics import (roc_auc_score, roc_curve, accuracy_score, 
                             matthews_corrcoef, precision_score, recall_score, 
                             f1_score, confusion_matrix)

c:\anaconda\envs\pytorch\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
#Model settings
batch_size = 32  # Setting batchsize
learning_rates=0.000065 # Setting learning rates
model_name="esm2_30"#Select different versions of the ESM2 model: esm2_6,esm_12,esm_30.

In [3]:
#Dataset definitions

class MyDataset(Dataset):
    def __init__(self, file):
        self.sequence, self.label = self.read_file(file)
        self.sequence_protbert=self.add_space_between_characters(self.sequence)

    def read_file(self,file_path):
        sequences = []
        labels = []
        with open(file_path, 'r', newline='') as csv_file:
            csv_reader = csv.reader(csv_file)
            next(csv_reader, None)  
            data = list(csv_reader)
            random.seed(42)
            random.shuffle(data)  
            for row in data:
                sequences.append(row[1])
                labels.append(row[2])
        return sequences, labels
    
    def add_space_between_characters(self,input_list):
        new_list = []
        for element in input_list:
            new_element = ' '.join(element)
            new_list.append(new_element)
        return new_list

    def __len__(self):
        return len(self.sequence)

    def __getitem__(self, index):
        sample=self.sequence[index]
        sample_protbert=self.sequence_protbert[index]
        label=int(self.label[index])
        return sample, label, sample_protbert

In [4]:
# Read the training set

train_file = 'E:\\LLM+XWT\\XWT数据\\Umami-BERT-UMP789\\1_1-data.csv'  
train_dataset = MyDataset(train_file)
train_dataloader = DataLoader(train_dataset, batch_size=batch_size)

In [5]:
#Define the esm family of models

class MyModel(nn.Module):
    def __init__(self,):
        super(MyModel, self).__init__()
        if model_name=="esm2_30":
            self.model = AutoModel.from_pretrained("facebook/esm2_t30_150M_UR50D")
            self.tokenizer = AutoTokenizer.from_pretrained("facebook/esm2_t30_150M_UR50D")
            self.layer=640
        elif model_name=="esm2_12":
            self.model = AutoModel.from_pretrained("facebook/esm2_t12_35M_UR50D")
            self.tokenizer = AutoTokenizer.from_pretrained("facebook/esm2_t12_35M_UR50D")
            self.layer=480
        elif model_name=="esm2_6":
            self.model = AutoModel.from_pretrained("facebook/esm2_t6_8M_UR50D")
            self.tokenizer = AutoTokenizer.from_pretrained("facebook/esm2_t6_8M_UR50D")
            self.layer=320
        self.fc2 = nn.Linear(self.layer, 2) 

    def forward(self, inputs,inputs2):
        inputs = self.tokenizer(inputs, padding=True, truncation=True, return_tensors="pt")
        input_ids = inputs["input_ids"].to(device)
        attention_mask = inputs["attention_mask"].to(device)
        outputs = self.model(input_ids=input_ids, attention_mask=attention_mask)
        pooler_output1 = outputs.pooler_output   
        x=self.fc2(pooler_output1)
        return x

In [6]:
#Model loading and setting

device = torch.device("cpu") 
random_seed = 42
random.seed(random_seed)
np.random.seed(random_seed)
torch.manual_seed(random_seed)
torch.cuda.manual_seed(random_seed)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
model = MyModel()
model.to(device)#Model loading
criterion = nn.CrossEntropyLoss()
loss_all=99999
best_auc=0
all_fpr = []
all_tpr = []
all_aucs = []

Some weights of EsmModel were not initialized from the model checkpoint at facebook/esm2_t30_150M_UR50D and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [7]:
# --- 2. 训练主循环 (核心修改部分) ---

# 数据加载 (请修改为你实际的路径)
train_file = 'E:\\LLM+XWT\\XWT数据\\Umami-BERT-UMP789\\1_1-data.csv' 
dataset = MyDataset(train_file) # 使用整个数据集，后面KFold会切分

from sklearn.model_selection import KFold
kf = KFold(n_splits=5, shuffle=False)

# 用于存储所有Fold的所有Epoch数据
all_folds_history = {} 
# 用于存储原本的ROC数据 (最佳Epoch)
roc_data = {'all_fpr': [], 'all_tpr': [], 'all_aucs': []}

epochs = 50 # 谢等人用了10个Epoch，你可以根据收敛情况调整，建议10-20即可

print(f"Start training model: {model_name}")

for fold, (train_indices, valid_indices) in enumerate(kf.split(dataset)):
    print(f"------ Fold {fold + 1} ------")
    
    # 初始化记录器
    history = {
        'train_loss': [], 'val_loss': [], 
        'val_auc': [], 'val_acc': [],
        'val_mcc': [], 'val_f1': [],
        'val_sn': [],  'val_sp': [], 'val_prec': []
    }
    
    train_sampler = SubsetRandomSampler(train_indices)
    valid_sampler = SubsetRandomSampler(valid_indices)
    train_dataloader = DataLoader(dataset, batch_size=batch_size, sampler=train_sampler)
    valid_dataloader = DataLoader(dataset, batch_size=batch_size, sampler=valid_sampler)
    
    model = MyModel()
    model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rates)
    criterion = nn.CrossEntropyLoss()
    
    best_auc_fold = 0
    best_fpr_fold = None
    best_tpr_fold = None
    
    for epoch in range(epochs):
        # --- Training Phase ---
        model.train()
        train_loss_accum = 0
        for batch_data, batch_labels, batch_prot in train_dataloader:
            batch_labels = batch_labels.to(device)
            # 注意：如果模型不需要protbert输入，只传batch_data
            outputs = model(batch_data, batch_prot) 
            loss = criterion(outputs, batch_labels)
            
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            
            train_loss_accum += loss.item()
            
        avg_train_loss = train_loss_accum / len(train_dataloader)
        
        # --- Validation Phase ---
        model.eval()
        val_loss_accum = 0
        all_labels = []
        all_probs = [] # 存储正类的概率
        all_preds = [] # 存储预测类别(0或1)
        
        with torch.no_grad():
            for batch_data, batch_labels, batch_prot in valid_dataloader:
                batch_labels = batch_labels.to(device)
                outputs = model(batch_data, batch_prot)
                loss = criterion(outputs, batch_labels)
                val_loss_accum += loss.item()
                
                probs = torch.softmax(outputs, dim=1)[:, 1] # 取label=1的概率
                preds = torch.argmax(outputs, dim=1)
                
                all_labels.extend(batch_labels.cpu().numpy())
                all_probs.extend(probs.cpu().numpy())
                all_preds.extend(preds.cpu().numpy())
        
        avg_val_loss = val_loss_accum / len(valid_dataloader)
        
        # 2. 计算所有指标 (ALL Metrics Calculation)
        # ------------------------------------------------------
        val_auc = roc_auc_score(all_labels, all_probs)
        val_acc = accuracy_score(all_labels, all_preds)
        val_mcc = matthews_corrcoef(all_labels, all_preds)
        val_f1 = f1_score(all_labels, all_preds)
        val_prec = precision_score(all_labels, all_preds, zero_division=0)
        val_sn = recall_score(all_labels, all_preds, zero_division=0) # Recall 就是 Sensitivity
        
        # 计算 Specificity (SP)
        tn, fp, fn, tp = confusion_matrix(all_labels, all_preds).ravel()
        val_sp = tn / (tn + fp) if (tn + fp) > 0 else 0
        # ------------------------------------------------------

        # 3. 保存所有指标到 history
        history['train_loss'].append(avg_train_loss)
        history['val_loss'].append(avg_val_loss)
        history['val_auc'].append(val_auc)
        history['val_acc'].append(val_acc)
        history['val_mcc'].append(val_mcc)
        history['val_f1'].append(val_f1)
        history['val_prec'].append(val_prec)
        history['val_sn'].append(val_sn)
        history['val_sp'].append(val_sp)
        
        print(f"Epoch {epoch+1}/{epochs} | Loss: {avg_train_loss:.4f} | AUC: {val_auc:.4f} | ACC: {val_acc:.4f} | MCC: {val_mcc:.4f}")
        

        # 保存最佳ROC数据
        if val_auc > best_auc_fold:
            best_auc_fold = val_auc
            fpr, tpr, _ = roc_curve(all_labels, all_probs)
            best_fpr_fold = fpr
            best_tpr_fold = tpr
            # 你也可以在这里计算并保存最佳Epoch的所有其他指标(MCC, SN, SP等)用于画Fig 3E
            
    # Fold结束，保存该Fold的ROC信息
    roc_data['all_fpr'].append(best_fpr_fold)
    roc_data['all_tpr'].append(best_tpr_fold)
    roc_data['all_aucs'].append(best_auc_fold)
    
    # 保存该Fold的历史记录
    all_folds_history[f'fold_{fold+1}'] = history

# --- 3. 保存所有数据 ---
save_dir = 'E:/LLM+XWT/最终版代码/训练结果/训练过程2_30/'
if not os.path.exists(save_dir):
    os.makedirs(save_dir)

# 保存完整训练记录（用于画谢等人的 Fig 3 A-D）
with open(os.path.join(save_dir, f'{model_name}_history.pkl'), 'wb') as f:
    pickle.dump(all_folds_history, f)

# 保存ROC数据（用于画你原来的 Fig 4）
with open(os.path.join(save_dir, f'{model_name}_roc_data.pkl'), 'wb') as f:
    pickle.dump(roc_data, f)
    
print("All data saved.")

Start training model: esm2_30
------ Fold 1 ------


Some weights of EsmModel were not initialized from the model checkpoint at facebook/esm2_t30_150M_UR50D and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


Epoch 1/50 | Loss: 0.6017 | AUC: 0.9289 | ACC: 0.8537 | MCC: 0.7259
Epoch 2/50 | Loss: 0.3167 | AUC: 0.9408 | ACC: 0.8374 | MCC: 0.7050
Epoch 3/50 | Loss: 0.2544 | AUC: 0.9564 | ACC: 0.8943 | MCC: 0.7887
Epoch 4/50 | Loss: 0.1618 | AUC: 0.9714 | ACC: 0.9187 | MCC: 0.8415
Epoch 5/50 | Loss: 0.1572 | AUC: 0.9688 | ACC: 0.8780 | MCC: 0.7792
Epoch 6/50 | Loss: 0.1366 | AUC: 0.9447 | ACC: 0.8537 | MCC: 0.7259
Epoch 7/50 | Loss: 0.0784 | AUC: 0.9651 | ACC: 0.9024 | MCC: 0.8204
Epoch 8/50 | Loss: 0.0552 | AUC: 0.9677 | ACC: 0.8780 | MCC: 0.7608
Epoch 9/50 | Loss: 0.0576 | AUC: 0.9662 | ACC: 0.8862 | MCC: 0.7872
Epoch 10/50 | Loss: 0.0294 | AUC: 0.9513 | ACC: 0.8862 | MCC: 0.7787
Epoch 11/50 | Loss: 0.0212 | AUC: 0.9685 | ACC: 0.8943 | MCC: 0.7887
Epoch 12/50 | Loss: 0.0109 | AUC: 0.9725 | ACC: 0.9106 | MCC: 0.8237
Epoch 13/50 | Loss: 0.0066 | AUC: 0.9662 | ACC: 0.8699 | MCC: 0.7432
Epoch 14/50 | Loss: 0.0129 | AUC: 0.9722 | ACC: 0.8943 | MCC: 0.7914
Epoch 15/50 | Loss: 0.0112 | AUC: 0.9648 | 

Some weights of EsmModel were not initialized from the model checkpoint at facebook/esm2_t30_150M_UR50D and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


Epoch 1/50 | Loss: 0.5556 | AUC: 0.9542 | ACC: 0.8699 | MCC: 0.7395
Epoch 2/50 | Loss: 0.2897 | AUC: 0.9719 | ACC: 0.9024 | MCC: 0.8047
Epoch 3/50 | Loss: 0.1503 | AUC: 0.9706 | ACC: 0.8943 | MCC: 0.7925
Epoch 4/50 | Loss: 0.0806 | AUC: 0.9621 | ACC: 0.9187 | MCC: 0.8385
Epoch 5/50 | Loss: 0.0890 | AUC: 0.9537 | ACC: 0.8780 | MCC: 0.7626
Epoch 6/50 | Loss: 0.0832 | AUC: 0.9584 | ACC: 0.8862 | MCC: 0.7742
Epoch 7/50 | Loss: 0.0679 | AUC: 0.9544 | ACC: 0.9106 | MCC: 0.8208
Epoch 8/50 | Loss: 0.0437 | AUC: 0.9372 | ACC: 0.9024 | MCC: 0.8077
Epoch 9/50 | Loss: 0.0247 | AUC: 0.9359 | ACC: 0.8943 | MCC: 0.7882
Epoch 10/50 | Loss: 0.0128 | AUC: 0.9359 | ACC: 0.8943 | MCC: 0.7888
Epoch 11/50 | Loss: 0.0280 | AUC: 0.9330 | ACC: 0.8699 | MCC: 0.7394
Epoch 12/50 | Loss: 0.0713 | AUC: 0.9319 | ACC: 0.8699 | MCC: 0.7394
Epoch 13/50 | Loss: 0.0395 | AUC: 0.9547 | ACC: 0.9024 | MCC: 0.8046
Epoch 14/50 | Loss: 0.0101 | AUC: 0.9452 | ACC: 0.9024 | MCC: 0.8047
Epoch 15/50 | Loss: 0.0029 | AUC: 0.9441 | 

...

Epoch 1/50 | Loss: 0.5614 | AUC: 0.8987 | ACC: 0.8293 | MCC: 0.6564
Epoch 2/50 | Loss: 0.3784 | AUC: 0.9419 | ACC: 0.8780 | MCC: 0.7549
Epoch 3/50 | Loss: 0.2506 | AUC: 0.9574 | ACC: 0.8862 | MCC: 0.7841
Epoch 4/50 | Loss: 0.1280 | AUC: 0.9691 | ACC: 0.9268 | MCC: 0.8524
Epoch 5/50 | Loss: 0.0715 | AUC: 0.9688 | ACC: 0.9268 | MCC: 0.8543
Epoch 6/50 | Loss: 0.0252 | AUC: 0.9638 | ACC: 0.9024 | MCC: 0.8043
Epoch 7/50 | Loss: 0.0066 | AUC: 0.9627 | ACC: 0.9024 | MCC: 0.8043
Epoch 8/50 | Loss: 0.0031 | AUC: 0.9616 | ACC: 0.8862 | MCC: 0.7708
Epoch 9/50 | Loss: 0.0355 | AUC: 0.9688 | ACC: 0.9024 | MCC: 0.8117
Epoch 10/50 | Loss: 0.0202 | AUC: 0.9630 | ACC: 0.8943 | MCC: 0.7867
Epoch 11/50 | Loss: 0.0062 | AUC: 0.9488 | ACC: 0.8780 | MCC: 0.7594
Epoch 12/50 | Loss: 0.0026 | AUC: 0.9555 | ACC: 0.8699 | MCC: 0.7408
Epoch 13/50 | Loss: 0.0806 | AUC: 0.9635 | ACC: 0.8862 | MCC: 0.7708
Epoch 14/50 | Loss: 0.0464 | AUC: 0.9688 | ACC: 0.9187 | MCC: 0.8369
Epoch 15/50 | Loss: 0.0560 | AUC: 0.9424 | 

Some weights of EsmModel were not initialized from the model checkpoint at facebook/esm2_t30_150M_UR50D and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


Epoch 1/50 | Loss: 0.5794 | AUC: 0.9675 | ACC: 0.8374 | MCC: 0.6982
Epoch 2/50 | Loss: 0.3089 | AUC: 0.9696 | ACC: 0.9187 | MCC: 0.8391
Epoch 3/50 | Loss: 0.1894 | AUC: 0.9725 | ACC: 0.9106 | MCC: 0.8212
Epoch 4/50 | Loss: 0.1035 | AUC: 0.9680 | ACC: 0.8943 | MCC: 0.7895
Epoch 5/50 | Loss: 0.0825 | AUC: 0.9733 | ACC: 0.9187 | MCC: 0.8374
Epoch 6/50 | Loss: 0.0781 | AUC: 0.9757 | ACC: 0.9350 | MCC: 0.8703
Epoch 7/50 | Loss: 0.0828 | AUC: 0.9728 | ACC: 0.9350 | MCC: 0.8699
Epoch 8/50 | Loss: 0.0355 | AUC: 0.9675 | ACC: 0.9106 | MCC: 0.8240
Epoch 9/50 | Loss: 0.0336 | AUC: 0.9651 | ACC: 0.8862 | MCC: 0.7741
Epoch 10/50 | Loss: 0.0850 | AUC: 0.9677 | ACC: 0.7724 | MCC: 0.6023
Epoch 11/50 | Loss: 0.0162 | AUC: 0.9588 | ACC: 0.9024 | MCC: 0.8052
Epoch 12/50 | Loss: 0.0753 | AUC: 0.9506 | ACC: 0.8780 | MCC: 0.7588
Epoch 13/50 | Loss: 0.0414 | AUC: 0.9598 | ACC: 0.8943 | MCC: 0.7887
Epoch 14/50 | Loss: 0.0061 | AUC: 0.9561 | ACC: 0.8943 | MCC: 0.7887
Epoch 15/50 | Loss: 0.0079 | AUC: 0.9566 | 

Some weights of EsmModel were not initialized from the model checkpoint at facebook/esm2_t30_150M_UR50D and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


Epoch 1/50 | Loss: 0.5167 | AUC: 0.8494 | ACC: 0.8033 | MCC: 0.6173
Epoch 2/50 | Loss: 0.2809 | AUC: 0.8866 | ACC: 0.7705 | MCC: 0.5398
Epoch 3/50 | Loss: 0.2030 | AUC: 0.9255 | ACC: 0.8689 | MCC: 0.7375
Epoch 4/50 | Loss: 0.1092 | AUC: 0.9042 | ACC: 0.8279 | MCC: 0.6540
Epoch 5/50 | Loss: 0.0553 | AUC: 0.9069 | ACC: 0.8361 | MCC: 0.6794
Epoch 6/50 | Loss: 0.0382 | AUC: 0.9039 | ACC: 0.8115 | MCC: 0.6227
Epoch 7/50 | Loss: 0.0486 | AUC: 0.9142 | ACC: 0.8115 | MCC: 0.6246
Epoch 8/50 | Loss: 0.0553 | AUC: 0.8972 | ACC: 0.8279 | MCC: 0.6562
Epoch 9/50 | Loss: 0.0413 | AUC: 0.9233 | ACC: 0.8443 | MCC: 0.6890
Epoch 10/50 | Loss: 0.0332 | AUC: 0.9341 | ACC: 0.8361 | MCC: 0.6766
Epoch 11/50 | Loss: 0.0107 | AUC: 0.9412 | ACC: 0.8607 | MCC: 0.7291
Epoch 12/50 | Loss: 0.0086 | AUC: 0.9339 | ACC: 0.8607 | MCC: 0.7219
Epoch 13/50 | Loss: 0.0053 | AUC: 0.9333 | ACC: 0.8607 | MCC: 0.7255
Epoch 14/50 | Loss: 0.0013 | AUC: 0.9347 | ACC: 0.8607 | MCC: 0.7229
Epoch 15/50 | Loss: 0.0008 | AUC: 0.9350 | 